# MiniCPM-OCR 1.0 — text, tables, and key fields on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/MiniCPM-OCR-1.0-document-ocr.ipynb)

[`openbmb/MiniCPM-OCR-1.0`](https://huggingface.co/openbmb/MiniCPM-OCR-1.0) is OpenBMB's **1.3B** OCR model, released October 2026 (Apache 2.0, not gated). It is the first OCR-specific model in the MiniCPM-V family: a SigLIP vision encoder plus a Qwen3.5 language model, trained on top of MiniCPM-V 4.6. The card reports 93.68 on Real5-OmniDocBench and 54.7 on OCRBench v2.

Why this model: it is small enough to sit next to anything else on a 16 GB GPU, and it answers **per-region prompts**. You send one crop plus a short instruction (`Text Recognition:`, `Table Recognition:`, `Formula Recognition:`, `Seal Recognition:`, `Chart to Table:`), or a **key-information-extraction** prompt that returns JSON. The architecture is native in **Transformers 5.21** (`MiniCPMOcrForConditionalGeneration`), so there is no `trust_remote_code`.

**Licence:** Apache 2.0.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has 16 GB of VRAM, no native bfloat16, and no FlashAttention 2. The weights are about **2.6 GB** in float16. A single page or crop with up to 1024 new tokens should stay well under 6 GB. That estimate comes from the weight size plus one page; it is not a measured Colab number yet.

**Not in this notebook:** the official `minicpmvocr` SDK (PP-DocLayoutV3 layout detection + Markdown assembly via PaddlePaddle), vLLM / SGLang serving, and DSpark speculative decoding. Those are separate environments on the card. Here you crop regions yourself and call the model directly.

**Links:** [model card](https://huggingface.co/openbmb/MiniCPM-OCR-1.0) · [GitHub](https://github.com/OpenBMB/MiniCPM-OCR) · [model guide](https://github.com/OpenBMB/MiniCPM-OCR/blob/main/docs/minicpm_ocr_1dot0_en.md)

In [ ]:
%pip install -q "transformers==5.21.0" accelerate pypdfium2 pillow

# MiniCPMOcrForConditionalGeneration (model_type "minicpm_ocr") ships in
# transformers 5.21.0, the version in the checkpoint's config.json.
# Colab's preinstalled torch is used as-is.
# Optional kernels (causal-conv1d, flash-linear-attention) are NOT installed:
# Transformers falls back to the reference PyTorch path for the Qwen3.5
# linear-attention layers. Same output, slower, and no CUDA build on Colab.
# pypdfium2 renders a PDF page for the table demo.

> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.

In [ ]:
import torch
import transformers
from packaging.version import Version

assert Version(transformers.__version__) >= Version("5.21.0"), (
    f"transformers {transformers.__version__} is loaded; MiniCPM-OCR 1.0 needs 5.21.0+. "
    "Run the install cell, then Runtime > Restart session."
)

print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

major, minor = torch.cuda.get_device_capability()
# T4 (sm_75) has no native bfloat16. The checkpoint is bf16; float16 is the T4 path.
DTYPE = torch.float16 if major < 8 else torch.bfloat16
props = torch.cuda.get_device_properties(0)
print("device:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
free, total = torch.cuda.mem_get_info()
print(f"VRAM free: {free / 1024**3:.1f} GB of {total / 1024**3:.1f} GB")
print("dtype:", DTYPE)
print("transformers", transformers.__version__, "| torch", torch.__version__)


def print_vram(label=""):
    suffix = f"  ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"VRAM allocated: {allocated:.2f} GB | peak: {peak:.2f} GB{suffix}")

## 1. Load MiniCPM-OCR 1.0

`AutoProcessor` + `AutoModelForImageTextToText`, native in Transformers 5.21. The first run downloads about 2.6 GB. SDPA attention is the default; the T4 has no FlashAttention 2.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "openbmb/MiniCPM-OCR-1.0"

torch.cuda.reset_peak_memory_stats()
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,
    attn_implementation="sdpa",
).to("cuda").eval()

n_params = sum(p.numel() for p in model.parameters())
print("class:", type(model).__name__)
print(f"parameters: {n_params / 1e9:.2f}B | dtype: {model.dtype}")
print_vram("after load")

## 2. Helpers and sample pages

* `ocr(image, prompt)` applies the chat template with `enable_thinking=False` (the card disables thinking for OCR; with it on, output can be empty or garbled), then greedy-decodes.
* `otsl_to_markdown` turns the model's table output (OTSL tags: `fcel` filled cell, `ecel` empty, `lcel` / `ucel` / `xcel` merged, `nl` new row) into a Markdown grid. The official SDK converts OTSL to HTML; this is a simpler teaching version.

Samples (same fixtures as the LightOnOCR / Unlimited-OCR-2 notebooks):

* the SROIE receipt from `hf-internal-testing/fixtures_ocr`
* a desk photo of a handwritten "Letter of Resignation" envelope ([CIPHR Connect](https://commons.wikimedia.org/wiki/File:Handwritten_Letter_of_Resignation.jpg), CC BY 2.0)
* page 8 of [Attention Is All You Need](https://arxiv.org/pdf/1706.03762) (arXiv:1706.03762), cropped to Table 2

In [ ]:
import io
import json
import re
import time

import pypdfium2 as pdfium
import requests
from IPython.display import Markdown, display
from PIL import Image

RECEIPT_URL = "https://huggingface.co/datasets/hf-internal-testing/fixtures_ocr/resolve/main/SROIE-receipt.jpeg"
# Mirror of the Wikimedia Commons file (CC BY 2.0); upload.wikimedia.org often returns 429 to Colab IPs.
LETTER_URLS = [
    "https://raw.githubusercontent.com/donvito/notebooks/main/colab/data/lightonocr/handwritten-letter-of-resignation.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/e/eb/Handwritten_Letter_of_Resignation.jpg",
]
PAPER_URL = "https://arxiv.org/pdf/1706.03762"
UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}
LONGEST = 1600  # T4-friendly cap on the longest edge


def fetch_bytes(url, attempts=4):
    for attempt in range(attempts):
        response = requests.get(url, headers=UA, timeout=120)
        if response.status_code not in (429, 500, 502, 503, 504) or attempt == attempts - 1:
            response.raise_for_status()
            return response.content
        time.sleep(2 * (attempt + 1))


def load_image(urls):
    urls = [urls] if isinstance(urls, str) else urls
    last = None
    for url in urls:
        try:
            image = Image.open(io.BytesIO(fetch_bytes(url))).convert("RGB")
            image.thumbnail((LONGEST, LONGEST))
            return image
        except Exception as exc:  # try the next mirror
            last = exc
    raise last


def ocr(image, prompt="Text Recognition:", max_new_tokens=1024):
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    ).to(model.device)
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(model.dtype)
    start = time.time()
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    text = processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()
    print(f"[{new_tokens.shape[1]} tokens in {time.time() - start:.1f}s]")
    print_vram()
    return text


def otsl_to_markdown(otsl):
    body = otsl.replace("<otsl>", "").replace("</otsl>", "")
    rows = []
    for raw_row in body.split("<nl>"):
        if not raw_row.strip():
            continue
        cells = []
        for tag, value in re.findall(r"<(fcel|ecel|lcel|ucel|xcel)>([^<]*)", raw_row):
            cells.append(value.strip() if tag == "fcel" else "")
        rows.append(cells)
    if not rows:
        return otsl
    width = max(len(r) for r in rows)
    rows = [r + [""] * (width - len(r)) for r in rows]
    lines = ["| " + " | ".join(rows[0]) + " |", "|" + "---|" * width]
    lines += ["| " + " | ".join(r) + " |" for r in rows[1:]]
    return "\n".join(lines)


receipt = load_image(RECEIPT_URL)
letter = load_image(LETTER_URLS)

pdf = pdfium.PdfDocument(fetch_bytes(PAPER_URL))
page8 = pdf[7].render(scale=1.6).to_pil().convert("RGB")
pdf.close()
w, h = page8.size
table_crop = page8.crop((0, int(h * 0.08), w, int(h * 0.42)))  # Table 2 sits near the top of page 8
print("receipt", receipt.size, "| letter", letter.size, "| table crop", table_crop.size)

### Receipt — `Text Recognition:`

The plain region prompt. The model returns the text in reading order.

In [ ]:
display(receipt)
receipt_text = ocr(receipt, "Text Recognition:", max_new_tokens=768)
print(receipt_text)

### Receipt — key information extraction (JSON)

This is the card's KIE prompt with our own field names. Fields not on the image come back as `"###"`. Always parse and check the JSON before you trust it downstream.

In [ ]:
FIELDS = ["store_name", "date", "total", "cashier"]
quoted = ", ".join(f'"{f}"' for f in FIELDS)
kie_prompt = (
    f"Extract the {quoted}. Format the output as a JSON object with keys: {quoted}. "
    'If an item is not in the image, use "###".'
)
kie_raw = ocr(receipt, kie_prompt, max_new_tokens=200)
print(kie_raw)

match = re.search(r"\{.*\}", kie_raw, re.S)
try:
    fields = json.loads(match.group(0)) if match else {}
except json.JSONDecodeError:
    fields = {}
for key in FIELDS:
    print(f"{key:>10}: {fields.get(key, '<missing>')}")

### Handwriting in a photo — `Text Recognition:`

A desk photo where the only text is a handwritten script line on an envelope. Expected answer: `Letter of Resignation`. This checks that the model reads cursive in a cluttered real-world capture and does not invent extra text.

In [ ]:
print("Source: CIPHR Connect, CC BY 2.0, via Wikimedia Commons.")
display(letter)
letter_text = ocr(letter, "Text Recognition:", max_new_tokens=200)
print(letter_text)

### Table — `Table Recognition:` on Table 2 of *Attention Is All You Need*

The model answers in OTSL. `otsl_to_markdown` renders it as a Markdown table (merged cells become blanks).

In [ ]:
display(table_crop)
table_otsl = ocr(table_crop, "Table Recognition:", max_new_tokens=1024)
print(table_otsl[:400], "..." if len(table_otsl) > 400 else "")
display(Markdown(otsl_to_markdown(table_otsl)))

## 3. Caveats and cleanup

**What this notebook is for.** The native Transformers path on a free T4: one crop + one region prompt per call, plus JSON key-field extraction. Use it to see the prompts and the raw output formats and to read allocated / peak VRAM.

**What it is not.** Full-page document parsing on the card goes through the `minicpmvocr` SDK: PP-DocLayoutV3 finds regions, the model reads each one, and the SDK assembles Markdown and converts OTSL to HTML. That needs PaddlePaddle in a separate environment from torch, so it is out of scope here. vLLM / SGLang / DSpark are serving and speed stories, not free-T4 teaching.

| Prompt | Use it for | Output |
| --- | --- | --- |
| `Text Recognition:` | Text blocks, receipts, handwriting | Plain text |
| `Table Recognition:` | One table crop | OTSL tags |
| `Formula Recognition:` | One formula crop | LaTeX |
| `Seal Recognition:` | Stamps / seals | Text |
| `Chart to Table:` | One chart | Table |
| KIE prompt (above) | Named fields | JSON with `"###"` for missing |

**T4 limits.** float16 (no native bfloat16), SDPA attention (no FlashAttention 2), and the reference PyTorch path for the linear-attention layers (no `causal-conv1d` / `flash-linear-attention` kernels). Images are capped at a 1600 px longest edge. If generation is slow on a long page, crop to the region you care about — that is how the model is meant to be used.

**Related notebooks:** [Unlimited-OCR-2](Unlimited-OCR-2-document-ocr.ipynb) (3.3B, page transcription), [LightOnOCR-3](LightOnOCR-3-0.8B-document-ocr.ipynb) (transcription + boxes), [OvisOCR3](OvisOCR3-structured-json.ipynb) (schema JSON), [Indra OCR](Indra-OCR-1M-document-ocr.ipynb) (CPU).

Free the GPU before you open another notebook:

In [ ]:
import gc

print_vram("before cleanup")
del model, processor
gc.collect()
torch.cuda.empty_cache()
print_vram("after cleanup")